
# 03 - Function Catalog

## Goal

Create the formal catalog of safe business functions that the Acme Motors Assistant is allowed to call.

The assistant will not generate SQL directly.

Instead, the flow will be:

User question  
↓  
Foundation model reads the function catalog  
↓  
Foundation model returns structured JSON  
↓  
Python validates the function name and arguments  
↓  
Python executes a safe SQL function  
↓  
Delta Table returns the real answer

## Example

User question:

¿Cuántas F150 vendí este mes?

↓

The function catalog tells the model that this type of question maps to:

get_product_sales_summary(product_model, date_range)

↓

Expected model output:

{
  "function_name": "get_product_sales_summary",
  "arguments": {
    "product_model": "F150",
    "date_range": "this_month"
  }
}

↓

Next step:

Python will execute this function against the auto_sales_transactions Delta table.

## Current step

We are here:

Validated SQL queries  
↓  
Function candidates  
↓  
Formal function catalog  
↓  
Later the foundation model uses this catalog to route questions

## Output of this notebook

Delta table:

assistant_function_catalog

In [0]:
import json
from pyspark.sql.functions import current_timestamp

SOURCE_TABLE = "auto_sales_transactions"
FUNCTION_CATALOG_TABLE = "assistant_function_catalog"

print(f"Source table: {SOURCE_TABLE}")
print(f"Function catalog table: {FUNCTION_CATALOG_TABLE}")

Source table: auto_sales_transactions
Function catalog table: assistant_function_catalog


In [0]:
candidates_df = spark.table("assistant_function_candidates")

display(candidates_df)

arguments,business_question,function_name,source_table
"List(product_model, date_range)",How many units and revenue for one vehicle model?,get_product_sales_summary,auto_sales_transactions
"List(date_range, limit)",What are the top selling vehicle models?,get_top_selling_models,auto_sales_transactions
List(date_range),Which sales representatives sold the most?,get_sales_by_rep,auto_sales_transactions
List(date_range),How much revenue was generated by state?,get_revenue_by_state,auto_sales_transactions
"List(product_model, date_range)",What is the average sale price for a vehicle model?,get_average_sale_price,auto_sales_transactions
List(date_range),How did each dealership perform?,get_dealership_sales_summary,auto_sales_transactions



## Argument validation

The model is allowed to choose a function, but it is not allowed to invent arbitrary arguments.

For this MVP, we will support these date ranges:

- this_month
- last_month
- year_to_date

And we will support vehicle models that exist in the Delta table.

Examples:

Valid:

{
  "product_model": "F150",
  "date_range": "this_month"
}

Invalid:

{
  "product_model": "spaceship",
  "date_range": "next century"
}

This is how we reduce hallucination risk.

In [0]:
# Get available models from Delta
available_models = [
    row["product_model"]
    for row in spark.table(SOURCE_TABLE)
        .select("product_model")
        .distinct()
        .orderBy("product_model")
        .collect()
]

available_models

['Accord',
 'Camry',
 'Civic',
 'Explorer',
 'F150',
 'Mustang',
 'RAV4',
 'Silverado']

In [0]:
# Define allowed date values

ALLOWED_DATE_RANGES = [
    "this_month",
    "last_month",
    "year_to_date"
]

DEFAULT_LIMIT = 5

print("Allowed date ranges:", ALLOWED_DATE_RANGES)
print("Available models:", available_models)

Allowed date ranges: ['this_month', 'last_month', 'year_to_date']
Available models: ['Accord', 'Camry', 'Civic', 'Explorer', 'F150', 'Mustang', 'RAV4', 'Silverado']


In [0]:
# Create Function catalog data

function_catalog = [
    {
        "function_name": "get_product_sales_summary",
        "description": (
            "Returns units sold, total revenue, and average sale price "
            "for a specific vehicle model in a date range."
        ),
        "arguments_schema_json": json.dumps({
            "product_model": {
                "type": "string",
                "required": True,
                "allowed_values": available_models
            },
            "date_range": {
                "type": "string",
                "required": True,
                "allowed_values": ALLOWED_DATE_RANGES
            }
        }),
        "example_questions_json": json.dumps([
            "How many F150 trucks did I sell this month?",
            "Cuántas F150 he vendido este mes?",
            "What revenue did Silverado generate last month?",
            "Dame el resumen de ventas de Mustang este mes"
        ]),
        "expected_json_example": json.dumps({
            "function_name": "get_product_sales_summary",
            "arguments": {
                "product_model": "F150",
                "date_range": "this_month"
            }
        }),
        "source_table": SOURCE_TABLE,
        "risk_control": "Model selects function only. SQL is executed by Python."
    },
    {
        "function_name": "get_top_selling_models",
        "description": (
            "Returns the top selling vehicle models by units sold and revenue "
            "for a date range."
        ),
        "arguments_schema_json": json.dumps({
            "date_range": {
                "type": "string",
                "required": True,
                "allowed_values": ALLOWED_DATE_RANGES
            },
            "limit": {
                "type": "integer",
                "required": False,
                "default": DEFAULT_LIMIT,
                "min": 1,
                "max": 10
            }
        }),
        "example_questions_json": json.dumps([
            "What are my top selling models this month?",
            "Cuáles son mis modelos más vendidos este mes?",
            "Show me the top 5 models year to date"
        ]),
        "expected_json_example": json.dumps({
            "function_name": "get_top_selling_models",
            "arguments": {
                "date_range": "this_month",
                "limit": 5
            }
        }),
        "source_table": SOURCE_TABLE,
        "risk_control": "Model selects ranking function. Python controls SQL and limit."
    },
    {
        "function_name": "get_sales_by_rep",
        "description": (
            "Returns units sold and revenue grouped by sales representative "
            "for a date range."
        ),
        "arguments_schema_json": json.dumps({
            "date_range": {
                "type": "string",
                "required": True,
                "allowed_values": ALLOWED_DATE_RANGES
            }
        }),
        "example_questions_json": json.dumps([
            "Which sales rep sold the most this month?",
            "Qué vendedor vendió más este mes?",
            "Show sales performance by representative year to date"
        ]),
        "expected_json_example": json.dumps({
            "function_name": "get_sales_by_rep",
            "arguments": {
                "date_range": "this_month"
            }
        }),
        "source_table": SOURCE_TABLE,
        "risk_control": "Model chooses aggregation. SQL grouping is predefined."
    },
    {
        "function_name": "get_revenue_by_state",
        "description": (
            "Returns units sold and revenue grouped by customer state "
            "for a date range."
        ),
        "arguments_schema_json": json.dumps({
            "date_range": {
                "type": "string",
                "required": True,
                "allowed_values": ALLOWED_DATE_RANGES
            }
        }),
        "example_questions_json": json.dumps([
            "How much revenue did we generate by state this month?",
            "Cuánto vendimos por estado este mes?",
            "Show revenue by state year to date"
        ]),
        "expected_json_example": json.dumps({
            "function_name": "get_revenue_by_state",
            "arguments": {
                "date_range": "this_month"
            }
        }),
        "source_table": SOURCE_TABLE,
        "risk_control": "Model chooses function. State aggregation query is controlled."
    },
    {
        "function_name": "get_average_sale_price",
        "description": (
            "Returns average, minimum, and maximum sale price for a specific "
            "vehicle model in a date range."
        ),
        "arguments_schema_json": json.dumps({
            "product_model": {
                "type": "string",
                "required": True,
                "allowed_values": available_models
            },
            "date_range": {
                "type": "string",
                "required": True,
                "allowed_values": ALLOWED_DATE_RANGES
            }
        }),
        "example_questions_json": json.dumps([
            "What was the average sale price for Mustang this month?",
            "Cuál fue el ticket promedio de Mustang este mes?",
            "Average sale price for F150 year to date"
        ]),
        "expected_json_example": json.dumps({
            "function_name": "get_average_sale_price",
            "arguments": {
                "product_model": "Mustang",
                "date_range": "this_month"
            }
        }),
        "source_table": SOURCE_TABLE,
        "risk_control": "Model cannot calculate price. SQL calculates AVG, MIN, and MAX."
    },
    {
        "function_name": "get_dealership_sales_summary",
        "description": (
            "Returns units sold, revenue, and average sale price grouped by dealership "
            "for a date range."
        ),
        "arguments_schema_json": json.dumps({
            "date_range": {
                "type": "string",
                "required": True,
                "allowed_values": ALLOWED_DATE_RANGES
            }
        }),
        "example_questions_json": json.dumps([
            "Show dealership sales this month",
            "Dame ventas por dealership este mes",
            "Which dealership generated the most revenue year to date?"
        ]),
        "expected_json_example": json.dumps({
            "function_name": "get_dealership_sales_summary",
            "arguments": {
                "date_range": "this_month"
            }
        }),
        "source_table": SOURCE_TABLE,
        "risk_control": "Model selects dealership summary. Python executes safe SQL."
    }
]

print(f"Functions defined: {len(function_catalog)}")

Functions defined: 6


In [0]:
# Create a DataFrame of the catalog
catalog_df = spark.createDataFrame(function_catalog)

catalog_df = catalog_df.withColumn("created_at", current_timestamp())

display(catalog_df)

arguments_schema_json,description,example_questions_json,expected_json_example,function_name,risk_control,source_table,created_at
"{""product_model"": {""type"": ""string"", ""required"": true, ""allowed_values"": [""Accord"", ""Camry"", ""Civic"", ""Explorer"", ""F150"", ""Mustang"", ""RAV4"", ""Silverado""]}, ""date_range"": {""type"": ""string"", ""required"": true, ""allowed_values"": [""this_month"", ""last_month"", ""year_to_date""]}}","Returns units sold, total revenue, and average sale price for a specific vehicle model in a date range.","[""How many F150 trucks did I sell this month?"", ""Cu\u00e1ntas F150 he vendido este mes?"", ""What revenue did Silverado generate last month?"", ""Dame el resumen de ventas de Mustang este mes""]","{""function_name"": ""get_product_sales_summary"", ""arguments"": {""product_model"": ""F150"", ""date_range"": ""this_month""}}",get_product_sales_summary,Model selects function only. SQL is executed by Python.,auto_sales_transactions,2026-05-24T06:55:25.156Z
"{""date_range"": {""type"": ""string"", ""required"": true, ""allowed_values"": [""this_month"", ""last_month"", ""year_to_date""]}, ""limit"": {""type"": ""integer"", ""required"": false, ""default"": 5, ""min"": 1, ""max"": 10}}",Returns the top selling vehicle models by units sold and revenue for a date range.,"[""What are my top selling models this month?"", ""Cu\u00e1les son mis modelos m\u00e1s vendidos este mes?"", ""Show me the top 5 models year to date""]","{""function_name"": ""get_top_selling_models"", ""arguments"": {""date_range"": ""this_month"", ""limit"": 5}}",get_top_selling_models,Model selects ranking function. Python controls SQL and limit.,auto_sales_transactions,2026-05-24T06:55:25.156Z
"{""date_range"": {""type"": ""string"", ""required"": true, ""allowed_values"": [""this_month"", ""last_month"", ""year_to_date""]}}",Returns units sold and revenue grouped by sales representative for a date range.,"[""Which sales rep sold the most this month?"", ""Qu\u00e9 vendedor vendi\u00f3 m\u00e1s este mes?"", ""Show sales performance by representative year to date""]","{""function_name"": ""get_sales_by_rep"", ""arguments"": {""date_range"": ""this_month""}}",get_sales_by_rep,Model chooses aggregation. SQL grouping is predefined.,auto_sales_transactions,2026-05-24T06:55:25.156Z
"{""date_range"": {""type"": ""string"", ""required"": true, ""allowed_values"": [""this_month"", ""last_month"", ""year_to_date""]}}",Returns units sold and revenue grouped by customer state for a date range.,"[""How much revenue did we generate by state this month?"", ""Cu\u00e1nto vendimos por estado este mes?"", ""Show revenue by state year to date""]","{""function_name"": ""get_revenue_by_state"", ""arguments"": {""date_range"": ""this_month""}}",get_revenue_by_state,Model chooses function. State aggregation query is controlled.,auto_sales_transactions,2026-05-24T06:55:25.156Z
"{""product_model"": {""type"": ""string"", ""required"": true, ""allowed_values"": [""Accord"", ""Camry"", ""Civic"", ""Explorer"", ""F150"", ""Mustang"", ""RAV4"", ""Silverado""]}, ""date_range"": {""type"": ""string"", ""required"": true, ""allowed_values"": [""this_month"", ""last_month"", ""year_to_date""]}}","Returns average, minimum, and maximum sale price for a specific vehicle model in a date range.","[""What was the average sale price for Mustang this month?"", ""Cu\u00e1l fue el ticket promedio de Mustang este mes?"", ""Average sale price for F150 year to date""]","{""function_name"": ""get_average_sale_price"", ""arguments"": {""product_model"": ""Mustang"", ""date_range"": ""this_month""}}",get_average_sale_price,"Model cannot calculate price. SQL calculates AVG, MIN, and MAX.",auto_sales_transactions,2026-05-24T06:55:25.156Z
"{""date_range"": {""type"": ""string"", ""required"": true, ""allowed_values"": [""this_month"", ""last_month"", ""year_to_date""]}}","Returns units sold, revenue, and average sale price grouped by dealership for a date range.","[""S

In [0]:
catalog_df.write.format("delta").mode("overwrite").saveAsTable(FUNCTION_CATALOG_TABLE)

print(f"Function catalog saved as Delta table: {FUNCTION_CATALOG_TABLE}")

Function catalog saved as Delta table: assistant_function_catalog


In [0]:
%sql
--Validate catalog
SELECT
  function_name,
  description,
  source_table,
  risk_control
FROM assistant_function_catalog
ORDER BY function_name;

function_name,description,source_table,risk_control
get_average_sale_price,"Returns average, minimum, and maximum sale price for a specific vehicle model in a date range.",auto_sales_transactions,"Model cannot calculate price. SQL calculates AVG, MIN, and MAX."
get_dealership_sales_summary,"Returns units sold, revenue, and average sale price grouped by dealership for a date range.",auto_sales_transactions,Model selects dealership summary. Python executes safe SQL.
get_product_sales_summary,"Returns units sold, total revenue, and average sale price for a specific vehicle model in a date range.",auto_sales_transactions,Model selects function only. SQL is executed by Python.
get_revenue_by_state,Returns units sold and revenue grouped by customer state for a date range.,auto_sales_transactions,Model chooses function. State aggregation query is controlled.
get_sales_by_rep,Returns units sold and revenue grouped by sales representative for a date range.,auto_sales_transactions,Model chooses aggregation. SQL grouping is predefined.
get_top_selling_models,Returns the top selling vehicle models by units sold and revenue for a date range.,auto_sales_transactions,Model selects ranking function. Python controls SQL and limit.


In [0]:
catalog_table = spark.table(FUNCTION_CATALOG_TABLE)

product_summary = (
    catalog_table
    .filter("function_name = 'get_product_sales_summary'")
    .collect()[0]
)

print("Function name:")
print(product_summary["function_name"])

print("\nDescription:")
print(product_summary["description"])

print("\nArguments schema:")
print(json.dumps(json.loads(product_summary["arguments_schema_json"]), indent=2))

print("\nExample questions:")
print(json.dumps(json.loads(product_summary["example_questions_json"]), indent=2, ensure_ascii=False))

print("\nExpected JSON:")
print(json.dumps(json.loads(product_summary["expected_json_example"]), indent=2))



Function name:
get_product_sales_summary

Description:
Returns units sold, total revenue, and average sale price for a specific vehicle model in a date range.

Arguments schema:
{
  "product_model": {
    "type": "string",
    "required": true,
    "allowed_values": [
      "Accord",
      "Camry",
      "Civic",
      "Explorer",
      "F150",
      "Mustang",
      "RAV4",
      "Silverado"
    ]
  },
  "date_range": {
    "type": "string",
    "required": true,
    "allowed_values": [
      "this_month",
      "last_month",
      "year_to_date"
    ]
  }
}

Example questions:
[
  "How many F150 trucks did I sell this month?",
  "Cuántas F150 he vendido este mes?",
  "What revenue did Silverado generate last month?",
  "Dame el resumen de ventas de Mustang este mes"
]

Expected JSON:
{
  "function_name": "get_product_sales_summary",
  "arguments": {
    "product_model": "F150",
    "date_range": "this_month"
  }
}



## Important distinction

The function catalog does not execute SQL.

The catalog is metadata.

It describes what functions exist and how the model should call them.

Execution happens later in the Function Registry.

Current responsibility:

Function Catalog  
→ describes available functions

Next responsibility:

Function Registry  
→ validates and executes the selected function

This separation matters because it makes the system easier to debug and safer.

In [0]:
# Create text for router prompt

def build_function_catalog_text(catalog_rows: list) -> str:
    blocks = []

    for row in catalog_rows:
        arguments_schema = json.loads(row["arguments_schema_json"])
        example_questions = json.loads(row["example_questions_json"])
        expected_json  = json.loads(row["expected_json_example"])

        block = f"""
        Function name:
{row["function_name"]}

Description:
{row["description"]}

Arguments schema:
{json.dumps(arguments_schema, indent=2)}

Example questions:
{json.dumps(example_questions, indent=2, ensure_ascii=False)}

Expected JSON output:
{json.dumps(expected_json, indent=2)}
"""

        blocks.append(block.strip())
        
    return "\n\n---\n\n".join(blocks)

catalog_rows = spark.table(FUNCTION_CATALOG_TABLE).collect()

catalog_prompt_text = build_function_catalog_text(catalog_rows)

print(catalog_prompt_text)

Function name:
get_product_sales_summary

Description:
Returns units sold, total revenue, and average sale price for a specific vehicle model in a date range.

Arguments schema:
{
  "product_model": {
    "type": "string",
    "required": true,
    "allowed_values": [
      "Accord",
      "Camry",
      "Civic",
      "Explorer",
      "F150",
      "Mustang",
      "RAV4",
      "Silverado"
    ]
  },
  "date_range": {
    "type": "string",
    "required": true,
    "allowed_values": [
      "this_month",
      "last_month",
      "year_to_date"
    ]
  }
}

Example questions:
[
  "How many F150 trucks did I sell this month?",
  "Cuántas F150 he vendido este mes?",
  "What revenue did Silverado generate last month?",
  "Dame el resumen de ventas de Mustang este mes"
]

Expected JSON output:
{
  "function_name": "get_product_sales_summary",
  "arguments": {
    "product_model": "F150",
    "date_range": "this_month"
  }
}

---

Function name:
get_top_selling_models

Description:
Retur

In [0]:
print(f"Catalog prompt length: {len(catalog_prompt_text)} characters")

Catalog prompt length: 4399 characters



# Notebook result

In this notebook we created the formal function catalog.

The assistant now has metadata for each allowed function:

- function name
- description
- argument schema
- allowed values
- example questions
- expected JSON output
- source Delta table
- risk control explanation

## Flow so far

Auto sales mock data  
↓  
auto_sales_transactions Delta table  
↓  
Manual SQL validation  
↓  
assistant_function_candidates  
↓  
assistant_function_catalog  

## Next notebook

Notebook 04 will create the Python Function Registry.

That registry will:

- receive function_name and arguments
- validate them
- execute the correct SQL query
- return a structured result

The foundation model will come after that.

In [0]:
%sql
SELECT
  function_name,
  description,
  source_table,
  risk_control
FROM assistant_function_catalog
ORDER BY function_name;

function_name,description,source_table,risk_control
get_average_sale_price,"Returns average, minimum, and maximum sale price for a specific vehicle model in a date range.",auto_sales_transactions,"Model cannot calculate price. SQL calculates AVG, MIN, and MAX."
get_dealership_sales_summary,"Returns units sold, revenue, and average sale price grouped by dealership for a date range.",auto_sales_transactions,Model selects dealership summary. Python executes safe SQL.
get_product_sales_summary,"Returns units sold, total revenue, and average sale price for a specific vehicle model in a date range.",auto_sales_transactions,Model selects function only. SQL is executed by Python.
get_revenue_by_state,Returns units sold and revenue grouped by customer state for a date range.,auto_sales_transactions,Model chooses function. State aggregation query is controlled.
get_sales_by_rep,Returns units sold and revenue grouped by sales representative for a date range.,auto_sales_transactions,Model chooses aggregation. SQL grouping is predefined.
get_top_selling_models,Returns the top selling vehicle models by units sold and revenue for a date range.,auto_sales_transactions,Model selects ranking function. Python controls SQL and limit.
